In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
import gradio as gr

In [5]:
OPENAI_KEY = os.getenv('OPENAI_API_KEY')
openai = OpenAI(api_key=OPENAI_KEY)

In [6]:
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    message = [{"role": "system", "content" : system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model = 'gpt-4.1-mini', messages=message)
    return response.choices[0].message.content

In [7]:
message_gpt("What is today's date?")

"Today's date is June 9, 2024."

In [ ]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()         

In [9]:
shout("Hello")

Shout has been called with input Hello


'HELLO'

In [16]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("jay", "momaya"))

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input dsadsa
Shout has been called with input hi
Shout has been called with input hi Jay


In [19]:
input_message = gr.Textbox(label="Your message", info="Enter a message to be shouted", lines=7)
output_message = gr.Textbox(label="Response", lines=10)

view = gr.Interface(
    fn=shout,
    title="Shout",
    inputs=[input_message],
    outputs=[output_message],
    flagging_mode="never"
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input Hi


In [20]:
input_message = gr.Textbox(label="Your message", info="Enter a message to be shouted", lines=7)
output_message = gr.Textbox(label="Response", lines=10)

view = gr.Interface(
    fn=message_gpt,
    title="Shout",
    inputs=[input_message],
    outputs=[output_message],
    flagging_mode="never"
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [21]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

input_message = gr.Textbox(label="Your message", info="Enter a message for GPT 4.1 mini", lines=7)
output_message = gr.Markdown(label="Response")

view = gr.Interface(
    fn=message_gpt,
    title="Shout",
    inputs=[input_message],
    outputs=[output_message],
    flagging_mode="never"
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [22]:
def stream_gpt(prompt):
    message = [
        {"role": "user", "content": prompt},
        {"role": "system", "content": system_message}
    ]
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=message,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [24]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

input_message = gr.Textbox(label="Your message", info="Enter a message for GPT 4.1 mini", lines=7)
output_message = gr.Markdown(label="Response")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT",
    inputs=[input_message],
    outputs=[output_message],
    flagging_mode="never"
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [25]:
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(base_url = ollama_url)

In [35]:
def ollama_stream(prompt):
    message = [
        {"role": "user", "content": prompt},
        {"role": "system", "content": system_message}
    ]
    stream = ollama.chat.completions.create(
        model="gpt-oss:latest",
        messages=message,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [32]:
!ollama pull gpt-oss:latest

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling e7b273f96360: 100% ▕██████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕██████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕██████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕██████████████████▏   18 B                         
pulling 776beb3adb23: 100% ▕██████████████████▏  489 B                         
verifying sha256 digest 
writing manifest ⠋ pulling manifest 
pulling e7b273f96360: 100% ▕██████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕██████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕██████████████████▏  11 KB                         
pullin

In [36]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

input_message = gr.Textbox(label="Your message", info="Enter a message for GPT 4.1 mini", lines=7)
output_message = gr.Markdown(label="Response")

view = gr.Interface(
    fn=ollama_stream,
    title="GPT",
    inputs=[input_message],
    outputs=[output_message],
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [37]:
def stream_model(prompt, model):
    if model.lower() == 'gpt':
        result = stream_gpt(prompt)
    elif model.lower() == 'ollama':
        result = ollama_stream(prompt)
    else:
        raise ValueError("Unknown Model")
    yield from result

In [40]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

input_message = gr.Textbox(label="Your message", info="Enter a message for GPT 4.1 mini", lines=7)
output_message = gr.Markdown(label="Response")
model_selector = gr.Dropdown(["GPT", "Ollama"], label="select a model", value="GPT")

view = gr.Interface(
    fn=stream_model,
    title="GPT",
    inputs=[input_message, model_selector],
    outputs=[output_message],
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.
